In [5]:
!unzip /home/student/nn_workshop/transfer-learning-workshop/data/hymenoptera_data.zip -d /home/student/nn_workshop/transfer-learning-workshop/data

Archive:  /home/student/nn_workshop/transfer-learning-workshop/data/hymenoptera_data.zip
replace /home/student/nn_workshop/transfer-learning-workshop/data/hymenoptera_data/train/ants/0013035.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: ^C


In [2]:
#1. Organize this script into a modular project using the following folder structure:

# transfer-learning-workshop/
#├── data/                  # Dataset folder (hymenoptera_data)
#├── checkpoints/           # Saved checkpoints
#├── src/
#│   ├── data_loader.py     # Dataloader functions
#    ├── model.py           # Model creation functions
#    ├── train.py           # Training and evaluation functions
#    └── main.py            # Script to run training
#├── eval.py                # Script to evaluate a saved checkpoint on test set
#└── requirements.txt       # Dependencies

#2.

#1. Organize this script into a modular project using the folder structure above:


#2. Move the corresponding code from the single-file script into the appropriate files:

    data_loader.py → dataset loading, splitting, and DataLoader creation

    model.py → model selection, freezing layers, and custom classifier

    train.py → training loop, evaluation function, checkpoint saving
  
    main.py → code to initialize everything and run training

    eval.py → code to load a checkpoint and evaluate on the test set

#3 Make the project CLI-friendly: allow choosing model (resnet18 or resnet34), batch size, learning rate, number of epochs, and checkpoint directory from the command line.

#4. Run your modularized project to verify that:

    Training and validation progress works

    Checkpoints are saved correctly

    Test evaluation reproduces the same accuracy as the single-file version

    command
    python workshop_single_file.py --data_dir data/hymenoptera_data --model_name resnet18 --epochs 5

    command to change model
    python workshop_single_file.py \
    --data_dir data/hymenoptera_data \
    --model_name resnet34 \
    --epochs 5

#5. Challenge: Finetune the model, unfreeze the last two layers 

In [6]:
"""
Transfer Learning Workshop

"""

import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
from tqdm import tqdm

# ----------------------------
# 1️⃣ Config (replaces argparse for Colab)
# ----------------------------
class Config:
    data_dir = "/home/student/nn_workshop/transfer-learning-workshop/data/"  # Update path if needed
    model_name = "resnet18"                       # resnet18 or resnet34
    batch_size = 32
    epochs = 5
    lr = 0.001
    feature_extract = True
    checkpoint_dir = "checkpoints"

args = Config()

# ----------------------------
# 2️⃣ Data Loader
# ----------------------------
def get_dataloaders(data_dir, batch_size=32):
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ])

    # Load train + val datasets
    train_dataset = datasets.ImageFolder(os.path.join(data_dir, "train"), transform=transform)
    val_dataset_original = datasets.ImageFolder(os.path.join(data_dir, "val"), transform=transform)

    # Merge datasets
    train_dataset.samples += val_dataset_original.samples
    train_dataset.targets += val_dataset_original.targets

    total_size = len(train_dataset)
    train_size = int(0.7 * total_size)
    val_size = int(0.15 * total_size)
    test_size = total_size - train_size - val_size

    train_set, val_set, test_set = random_split(
        train_dataset, [train_size, val_size, test_size],
        generator=torch.Generator().manual_seed(42)
    )

    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, test_loader

# ----------------------------
# 3️⃣ Model
# ----------------------------
def freeze_layers(model):
    for param in model.parameters():
        param.requires_grad = false

def get_model(model_name="resnet18", num_classes=2, feature_extract=True):
    if model_name=="resnet18":
        model = models.resnet18(weights="IMAGENET1K_V1")
    elif model_name=="resnet34":
        model = models.resnet34(weights="IMAGENET1K_V1")
    else:
        raise ValueError("Supported models: resnet18, resnet34")
    if feature_extract:
        freeze_layers(model)
    num_features = model.fc.in_features
    model.fc = nn.Linear(num_features, num_classes)
    return model

# ----------------------------
# 4️⃣ Train / Evaluate
# ----------------------------
def evaluate(model, loader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs , labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs,1)
            total += labels.size(0)
            correct += (preds==labels).sum().item()
    return correct/total

def train(model, train_loader, val_loader, criterion, optimizer,
          device, epochs, checkpoint_dir, model_name):
    os.makedirs(checkpoint_dir, exist_ok=True)
    model.to(device)

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        val_acc = evaluate(model, val_loader, device)
        print(f"\nEpoch {epoch+1} | Loss: {running_loss:.4f} | Val Acc: {val_acc:.4f}")
        checkpoint_path = os.path.join(checkpoint_dir, f"{model_name}_epoch_{epoch+1}.pth")
        torch.save({
            "epoch": epoch+1,
            "model_name": model_name,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_accuracy": val_acc
        }, checkpoint_path)
        print(f"Checkpoint saved at: {checkpoint_path}")

# ----------------------------
# 5️⃣ Main
# ----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_loader, val_loader, test_loader = get_dataloaders(args.data_dir, args.batch_size)
model = get_model(args.model_name, num_classes=2, feature_extract=args.feature_extract)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=args.lr)

print("Starting Training...\n")
train(model, train_loader, val_loader, criterion, optimizer,
      device, args.epochs, args.checkpoint_dir, args.model_name)

print("\nEvaluating on Test Set...")
test_acc = evaluate(model, test_loader, device)
print(f"Test Accuracy: {test_acc:.4f}")

FileNotFoundError: [Errno 2] No such file or directory: '/home/student/nn_workshop/transfer-learning-workshop/data/train'